# 8 · Feature Engineering & GroupBy
*Intro to Python for Scientists & Public Health Professionals*

Two everyday skills: **building new columns** from existing ones, and **summarizing by group** with the split-apply-combine pattern. We'll work on the diabetes dataset, applying a compact version of the cleaning from Lesson 7.

### By the end of this notebook you can
- Create new columns with vectorized math, `.map`, and `apply`
- Bin a numeric column into categories with `cut` / `qcut`
- Group by one or more categories and aggregate
- Use `.agg` with named outputs, and `transform` to add group stats back to rows

### Agenda
1. New columns (feature engineering)
2. GroupBy basics
3. Aggregating with `.agg`
4. Grouping by multiple categories
5. Binning, transform, filter

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:


BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_inspect.csv")

# Compact clean (the full walkthrough is in Lesson 7)
df = df.drop_duplicates().replace("?", pd.NA)
df["gender"] = (df["gender"].str.strip().str.lower()
                  .replace({"m": "male", "f": "female", "mle": "male", "unknown/invalid": pd.NA}))
df = df[df["age"] != "xyz"]
df["num_medications"] = df["num_medications"].fillna(df["num_medications"].median())
df.shape

## 1. New columns (feature engineering)

The fastest way is **vectorized** arithmetic on existing columns — no loop.

In [ ]:
df["total_procedures"] = df["num_procedures"] + df["num_lab_procedures"]   # combine two columns
df["meds_per_day"] = df["num_medications"] / df["time_in_hospital"]         # a ratio feature
df[["num_procedures", "num_lab_procedures", "total_procedures", "meds_per_day"]].head()

Recode a categorical to numbers with `.map` (pass a dict of replacements).

Keep the mental rule: .replace() when unmatched should survive, .map() when unmatched should die.

In [ ]:
df["gender_code"] = df["gender"].map({"male": 0, "female": 1})
df[["gender", "gender_code"]].head()

# If there was an 'm' in the column .map will change it to NaN

For row-wise logic, `apply(..., axis=1)` runs a function on each row. It's flexible but slow on large data — when the logic is a simple condition, a **vectorized** `np.where` does the same thing far faster.

In [ ]:
# Flexible but slower: a function applied row by row
def stay_label(row):
    return "long" if row["time_in_hospital"] > 7 else "short"

df["stay_flag"] = df.apply(stay_label, axis=1)

# Same result, vectorized and much faster:
df["stay_flag"] = np.where(df["time_in_hospital"] > 7, "long", "short")
df[["time_in_hospital", "stay_flag"]].head()

## 2. GroupBy basics

The mental model is **split-apply-combine**: split the rows into groups by some key, apply an aggregation to each group, and combine the results into one table.

In [ ]:
df.groupby("gender")["num_medications"].mean()    # one number per group

In [ ]:
df.groupby("gender").size()                       # how many rows in each group

Aggregating *all* numeric columns at once needs `numeric_only=True`, because the frame also has text columns that can't be averaged.

In [ ]:
df.groupby("gender").mean(numeric_only=True)

> Reference: [GroupBy user guide](https://pandas.pydata.org/docs/user_guide/groupby.html).

## 3. Aggregating with `.agg`

`.agg` runs several aggregations at once. Use **named aggregations** for clean output column names.

In [ ]:
df.groupby("gender")["num_medications"].agg(["mean", "median", "min", "max", "count"])

In [ ]:
df.groupby("gender").agg(
    avg_meds=("num_medications", "mean"),
    avg_stay=("time_in_hospital", "mean"),
    n=("encounter_id", "size"),
)

### Exercise 1 — Summarize by group *(8 min)*

1. For each `gender`, find the mean `time_in_hospital` and the number of encounters.
2. Which `age` band has the highest mean `num_medications`?

In [ ]:
# Your work here


## 4. Grouping by multiple categories

Pass a **list** of keys. The result has a multi-level index; `as_index=False` (or `.reset_index()`) flattens it back to columns.

In [ ]:
df.groupby(["gender", "age"])["time_in_hospital"].mean().head(8)

In [ ]:
df.groupby(["gender", "age"], as_index=False)["time_in_hospital"].mean().head()

## 5. Binning, transform, filter

Turn a numeric column into categories with `cut` (fixed edges) or `qcut` (equal-sized quantile bins) — a common feature-engineering step that pairs naturally with grouping.

In [ ]:
df["stay_band"] = pd.cut(df["time_in_hospital"], bins=[0, 3, 7, 14],
                         labels=["short", "medium", "long"])
df.groupby("stay_band", observed=True)["num_medications"].mean()

In [ ]:
pd.qcut(df["num_medications"], 4).value_counts().sort_index()   # four equal-sized bins

`transform` is like `agg` but returns a value **for every row** (aligned to the original), so you can attach a group statistic back onto each record.

In [ ]:
df["gender_avg_stay"] = df.groupby("gender")["time_in_hospital"].transform("mean")
df[["gender", "time_in_hospital", "gender_avg_stay"]].head()

`filter` keeps or drops whole groups based on a condition about the group.

In [ ]:
# keep only age bands with at least 5,000 encounters
df.groupby("age").filter(lambda g: len(g) >= 5000).shape

### Capstone Part 2

Open your **Rural Hospital Closures capstone** and complete **Part 2 (GroupBy)**: closures per state, mean beds by closure type, counts and mean beds by payment type, and closures per year.

## Wrap-up

You can engineer new columns (vectorized math, `.map`, `apply`, `cut`/`qcut`), and summarize data with groupby, `.agg`, multi-key grouping, `transform`, and `filter`.

**Next:** Working with dates and times.